# Laboratório — Poisson e exponencial

Este laboratório reproduz os cálculos da Aula 08 e testa as hipóteses com simulação. Trabalharemos com contagens, esperas, aproximação binomial–Poisson, ausência de memória e sobredispersão.

**Dependências:** Python ≥ 3.10, NumPy ≥ 1.26, SciPy ≥ 1.11 e Matplotlib ≥ 3.8.  
**Reprodutibilidade:** todas as simulações usam `numpy.random.default_rng(20260907)`.


## 1. Ambiente e funções auxiliares

A SciPy parametriza `poisson` pela média `mu`. Para `expon`, usa a escala; se a taxa é $r$, então `scale=1/r`. A função auxiliar abaixo exige taxa e exposição positivas e evita misturar essas grandezas silenciosamente.


In [1]:
import sys
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import poisson, expon, binom

SEED = 20260907
rng = np.random.default_rng(SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__} | SciPy: {scipy.__version__} | Matplotlib: {matplotlib.__version__}")
print(f"Seed: {SEED}")

Python: 3.12.13
NumPy: 2.3.5 | SciPy: 1.17.0 | Matplotlib: 3.10.8
Seed: 20260907


In [2]:
def media_da_janela(taxa, exposicao):
    """Converte taxa por unidade e exposição na média Poisson da janela."""
    if taxa <= 0 or exposicao <= 0:
        raise ValueError("taxa e exposição devem ser positivas")
    return taxa * exposicao

assert media_da_janela(3.0, 2.0) == 6.0
try:
    media_da_janela(3.0, -1.0)
except ValueError:
    print("Validação de entrada: OK")

Validação de entrada: OK


## 2. Poisson no exemplo motivador

O serviço recebe 3 requisições por minuto. Em 2 minutos, a média da janela é $\lambda=6$. Calcularemos $P(N=4)$ e $P(N\ge1)$. Para a segunda, usamos `sf(0)`, isto é, a probabilidade de $N>0$.


In [3]:
taxa = 3.0       # requisições/minuto
exposicao = 2.0  # minutos
lam = media_da_janela(taxa, exposicao)

p_exatamente_4 = poisson.pmf(4, mu=lam)
p_ao_menos_1 = poisson.sf(0, mu=lam)

print(f"lambda da janela: {lam:.1f}")
print(f"P(N=4): {p_exatamente_4:.9f}")
print(f"P(N>=1): {p_ao_menos_1:.9f}")

assert np.isclose(p_exatamente_4, 0.1338526175, atol=1e-10)
assert np.isclose(p_ao_menos_1, 0.9975212478, atol=1e-10)

lambda da janela: 6.0
P(N=4): 0.133852618
P(N>=1): 0.997521248


## 3. Simulação de contagens

Geramos 200 mil janelas independentes. A média e a variância amostrais devem ficar próximas de 6; as barras empíricas devem acompanhar a PMF teórica. Pequenas diferenças são erro de Monte Carlo, não falha do modelo.


In [4]:
n_sim = 200_000
contagens = rng.poisson(lam=lam, size=n_sim)
media_emp = contagens.mean()
var_emp = contagens.var(ddof=0)

print(f"Média empírica: {media_emp:.6f} | teórica: {lam:.6f}")
print(f"Variância empírica: {var_emp:.6f} | teórica: {lam:.6f}")

assert abs(media_emp - lam) < 0.03
assert abs(var_emp - lam) < 0.08

Média empírica: 5.997235 | teórica: 6.000000
Variância empírica: 5.980347 | teórica: 6.000000


In [5]:
ks = np.arange(0, 16)
freq_emp = np.array([(contagens == k).mean() for k in ks])
pmf_teorica = poisson.pmf(ks, mu=lam)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(ks - 0.18, freq_emp, width=0.36, label="Simulação", color="#2563eb")
ax.bar(ks + 0.18, pmf_teorica, width=0.36, label="PMF teórica", color="#f59e0b")
ax.set(xlabel="Requisições na janela", ylabel="Probabilidade", title="Poisson(6): simulação × teoria")
ax.set_xticks(ks)
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.show()

assert np.max(np.abs(freq_emp - pmf_teorica)) < 0.004

## 4. A exposição muda a distribuição

A taxa permanece 3 por minuto, mas a média Poisson cresce linearmente com a duração. Isso desloca a massa para contagens maiores.


In [6]:
duracoes = np.array([0.5, 1.0, 2.0])
lambdas = taxa * duracoes

for t, valor in zip(duracoes, lambdas):
    print(f"{t:>3.1f} min -> lambda={valor:.1f}")

fig, ax = plt.subplots(figsize=(9, 4.5))
grade_k = np.arange(0, 16)
for t, valor in zip(duracoes, lambdas):
    ax.plot(grade_k, poisson.pmf(grade_k, valor), marker="o", label=f"t={t:g} min; λ={valor:g}")
ax.set(xlabel="Contagem k", ylabel="P(N=k)", title="Mesma taxa, exposições diferentes")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

assert np.allclose(lambdas, [1.5, 3.0, 6.0])

0.5 min -> lambda=1.5
1.0 min -> lambda=3.0
2.0 min -> lambda=6.0


## 5. Aproximação binomial–Poisson

Com 100 oportunidades independentes e probabilidade 0,03, a binomial tem média 3. Compararemos sua PMF com Poisson(3). A distância de variação total, metade da soma das diferenças absolutas, resume o desvio entre as distribuições.


In [7]:
n, p = 100, 0.03
lam_aprox = n * p
k_aprox = np.arange(0, n + 1)
pmf_bin = binom.pmf(k_aprox, n=n, p=p)
pmf_pois = poisson.pmf(k_aprox, mu=lam_aprox)
dist_tv = 0.5 * np.abs(pmf_bin - pmf_pois).sum()

print(f"P_bin(X=2): {binom.pmf(2, n, p):.9f}")
print(f"P_pois(Y=2): {poisson.pmf(2, lam_aprox):.9f}")
print(f"Distância de variação total: {dist_tv:.6f}")

assert np.isclose(poisson.pmf(2, lam_aprox), 0.2240418077, atol=1e-10)
assert dist_tv < 0.02
assert np.isclose(pmf_bin.sum(), 1.0, atol=1e-12)
assert np.isclose(pmf_pois.sum(), 1.0, atol=1e-12)

P_bin(X=2): 0.225152963
P_pois(Y=2): 0.224041808
Distância de variação total: 0.007607


## 6. Esperas exponenciais

Para taxa 3 por minuto, calculamos a chance de esperar mais de 30 segundos e a de receber uma requisição em até 20 segundos. Tempos são convertidos para minutos antes do cálculo.


In [8]:
r = 3.0  # por minuto
escala = 1 / r
t_30s = 30 / 60
t_20s = 20 / 60

p_mais_30s = expon.sf(t_30s, scale=escala)
p_ate_20s = expon.cdf(t_20s, scale=escala)

print(f"Escala = 1/r: {escala:.6f} minuto")
print(f"P(T>30 s): {p_mais_30s:.9f}")
print(f"P(T<=20 s): {p_ate_20s:.9f}")

assert np.isclose(p_mais_30s, np.exp(-1.5), atol=1e-12)
assert np.isclose(p_ate_20s, 1 - np.exp(-1), atol=1e-12)

Escala = 1/r: 0.333333 minuto
P(T>30 s): 0.223130160
P(T<=20 s): 0.632120559


### Simulação das esperas

A média teórica é $1/r=1/3$ minuto e a variância é $1/r^2=1/9$ minuto². O histograma deve acompanhar a densidade, mas cada barra representa uma faixa, não uma probabilidade pontual.


In [9]:
esperas = rng.exponential(scale=escala, size=n_sim)
media_espera = esperas.mean()
var_espera = esperas.var(ddof=0)

print(f"Média empírica: {media_espera:.6f} | teórica: {escala:.6f}")
print(f"Variância empírica: {var_espera:.6f} | teórica: {escala**2:.6f}")

assert abs(media_espera - escala) < 0.003
assert abs(var_espera - escala**2) < 0.004

Média empírica: 0.333311 | teórica: 0.333333
Variância empírica: 0.111264 | teórica: 0.111111


In [10]:
x = np.linspace(0, 2, 400)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(esperas, bins=70, range=(0, 2), density=True, alpha=0.55, color="#2563eb", label="Simulação")
ax.plot(x, expon.pdf(x, scale=escala), color="#dc2626", linewidth=2.5, label="PDF teórica")
ax.set(xlabel="Espera (minutos)", ylabel="Densidade", title="Exponencial com taxa 3/min")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 7. Ausência de memória

Verificaremos $P(T>s+t\mid T>s)=P(T>t)$ de duas formas: exatamente pela sobrevivência e aproximadamente pela amostra. Escolhemos $s=0,4$ e $t=0,3$ minuto.


In [11]:
s, t = 0.4, 0.3
cond_teorica = expon.sf(s + t, scale=escala) / expon.sf(s, scale=escala)
direta_teorica = expon.sf(t, scale=escala)

cond_empirica = np.mean(esperas > s + t) / np.mean(esperas > s)
direta_empirica = np.mean(esperas > t)

print(f"Condicional teórica: {cond_teorica:.6f} | direta teórica: {direta_teorica:.6f}")
print(f"Condicional empírica: {cond_empirica:.6f} | direta empírica: {direta_empirica:.6f}")

assert np.isclose(cond_teorica, direta_teorica, atol=1e-12)
assert abs(cond_empirica - direta_teorica) < 0.01
assert abs(direta_empirica - direta_teorica) < 0.005

Condicional teórica: 0.406570 | direta teórica: 0.406570
Condicional empírica: 0.405674 | direta empírica: 0.406905


## 8. Uma trajetória do processo de Poisson

Geramos intervalos exponenciais e acumulamos até 10 minutos. A curva em degraus cresce uma unidade a cada chegada. Uma trajetória isolada não precisa terminar exatamente em $rt=30$; 30 é a contagem esperada.


In [12]:
rng_traj = np.random.default_rng(SEED + 1)
horizonte = 10.0
intervalos = rng_traj.exponential(scale=1 / r, size=200)
chegadas = np.cumsum(intervalos)
chegadas = chegadas[chegadas <= horizonte]

tempos_degrau = np.r_[0, chegadas, horizonte]
contagens_degrau = np.r_[0, np.arange(1, len(chegadas) + 1), len(chegadas)]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.step(tempos_degrau, contagens_degrau, where="post", color="#7c3aed")
ax.scatter(chegadas, np.arange(1, len(chegadas) + 1), s=18, color="#7c3aed", label="Chegadas")
ax.axhline(r * horizonte, color="#f59e0b", linestyle="--", label="Contagem esperada em 10 min")
ax.set(xlabel="Tempo (minutos)", ylabel="Contagem acumulada", title="Trajetória simulada por intervalos exponenciais")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

print(f"Eventos observados: {len(chegadas)} | valor esperado: {r*horizonte:.1f}")
assert np.all(np.diff(chegadas) > 0)
assert np.all(chegadas <= horizonte)

Eventos observados: 33 | valor esperado: 30.0


## 9. Sobredispersão por mistura de taxas

Com taxa constante 3, a variância das contagens fica próxima da média. Agora alternaremos aleatoriamente entre janelas de taxa 1 e 5. A taxa média ainda é 3, mas a heterogeneidade aumenta a variância populacional para $E[\lambda]+Var(\lambda)=3+4=7$.


In [13]:
rng_disp = np.random.default_rng(SEED + 2)
n_janelas = 200_000

constante = rng_disp.poisson(3.0, size=n_janelas)
taxas_mistas = rng_disp.choice([1.0, 5.0], size=n_janelas)
mistura = rng_disp.poisson(taxas_mistas)

def resumo_disp(nome, valores):
    media = valores.mean()
    variancia = valores.var(ddof=0)
    print(f"{nome:13s} média={media:.4f} variância={variancia:.4f} índice={variancia/media:.3f}")
    return media, variancia

m_const, v_const = resumo_disp("Poisson(3)", constante)
m_mix, v_mix = resumo_disp("Mistura 1/5", mistura)

assert abs(m_const - 3.0) < 0.02
assert abs(v_const - 3.0) < 0.04
assert abs(m_mix - 3.0) < 0.03
assert abs(v_mix - 7.0) < 0.10
assert v_mix / m_mix > 2.2

Poisson(3)    média=3.0045 variância=3.0033 índice=1.000
Mistura 1/5   média=3.0027 variância=7.0097 índice=2.334


## 10. Checagens finais

As asserções abaixo condensam invariantes metodológicos: PMFs válidas, média/variância teóricas, unidade consistente e correspondência entre contagem e espera.


In [14]:
assert np.isclose(poisson.mean(lam), lam)
assert np.isclose(poisson.var(lam), lam)
assert np.isclose(expon.mean(scale=1/r), 1/r)
assert np.isclose(expon.var(scale=1/r), 1/r**2)
assert np.isclose(poisson.pmf(np.arange(0, 50), lam).sum(), 1.0, atol=1e-12)
assert np.isclose(expon.sf(0.5, scale=1/r), np.exp(-r * 0.5))

print("Todas as verificações numéricas foram concluídas com sucesso.")

Todas as verificações numéricas foram concluídas com sucesso.


## Conclusões

- A média Poisson da janela é taxa × exposição; declarar unidades evita erros.
- Média e variância das contagens simuladas se aproximaram de $\lambda$.
- A Poisson aproximou bem a binomial do exemplo, mas o erro foi medido em vez de presumido.
- Na SciPy, a exponencial recebe `scale=1/r`.
- A ausência de memória apareceu na teoria e na simulação.
- Misturar taxas produziu sobredispersão mesmo mantendo a mesma média global.

Próximo passo: estudar a distribuição normal, z-score e normal multivariada na Aula 09.
